#  **Data Collection and Preprocessing**

### Import Libraries

In [1]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

In [2]:
import pandas as pd
import numpy as np
from google_play_scraper import app, Sort, reviews
import re
from datetime import datetime
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    scrap_reviews,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    rating_distribution,
    preprocessing_report,
    save_cleaned_data,
)

### Web Scraping

#### App metadata

In [3]:
CBE_APP_ID = 'com.combanketh.mobilebanking'
display_app_info(CBE_APP_ID)

CBE App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.2864957
Total Ratings: 48,302
Total Reviews: 9,309
Installs     : 5,000,000+


#### Scrape reviews

In [4]:
reviews = scrap_reviews(app_id=CBE_APP_ID)

Scraping reviews for com.combanketh.mobilebanking...
Collected 700 raw reviews


#### Collect review text, rating, review date, bank , source

In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: eda74236-f7f3-4422-a139-a181e832bc27
  userName: Usman Mohammed
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocJu6_G-y5BAGIYJKukXNxVDSPWLdbQZbTmOcf1o80ROo9oHIQ=mo
  content: thank you cbe
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-13 17:16:37
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [6]:
df = review_dataframe(reviews)

print(f"Shape: {df.shape}")
df.head()

Shape: (700, 6)


,review_id,review,rating,date,bank,source
0,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,CBE Bank,Google Play
1,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,CBE Bank,Google Play
2,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13 12:19:17,CBE Bank,Google Play
3,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13 11:27:52,CBE Bank,Google Play
4,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",5,2026-05-13 11:21:48,CBE Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [7]:
remove_duplicates(df)

Removed 0 duplicate reviews
Remaining: 700 reviews


,review_id,review,rating,date,bank,source
0,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,CBE Bank,Google Play
1,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,CBE Bank,Google Play
2,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13 12:19:17,CBE Bank,Google Play
3,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13 11:27:52,CBE Bank,Google Play
4,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",5,2026-05-13 11:21:48,CBE Bank,Google Play
...,...,...,...,...,...,...
695,fb955d85-3a17-4c39-80c8-a9d967ecab8f,"this is the worst update, what have you done w...",1,2026-02-03 20:54:11,CBE Bank,Google Play
696,947f41d3-8d10-4a9b-b499-785400609a39,how to open,5,2026-02-03 20:36:24,CBE Bank,Google Play
697,9e80832f-cddb-446d-891a-0cc61044cf83,I very much thanks again and again.,5,2026-02-03 20:02:58,CBE Bank,Google Play
698,918fc1bc-8054-4699-8f31-971e5a19930e,እየሰራ አይደለም,1,2026-02-03 19:45:36,CBE Bank,Google Play


#### Handle missing values

In [8]:
handle_missing_data(df)

Removed 0 rows with missing critical data
Remaining: 700 reviews


,review_id,review,rating,date,bank,source
0,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,CBE Bank,Google Play
1,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,CBE Bank,Google Play
2,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13 12:19:17,CBE Bank,Google Play
3,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13 11:27:52,CBE Bank,Google Play
4,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",5,2026-05-13 11:21:48,CBE Bank,Google Play
...,...,...,...,...,...,...
695,fb955d85-3a17-4c39-80c8-a9d967ecab8f,"this is the worst update, what have you done w...",1,2026-02-03 20:54:11,CBE Bank,Google Play
696,947f41d3-8d10-4a9b-b499-785400609a39,how to open,5,2026-02-03 20:36:24,CBE Bank,Google Play
697,9e80832f-cddb-446d-891a-0cc61044cf83,I very much thanks again and again.,5,2026-02-03 20:02:58,CBE Bank,Google Play
698,918fc1bc-8054-4699-8f31-971e5a19930e,እየሰራ አይደለም,1,2026-02-03 19:45:36,CBE Bank,Google Play


#### Normalize dates to YYYY-MM-DD format

In [9]:
normalize_dates(df)

Before normalization:
0   2026-05-13 17:16:37
1   2026-05-13 16:18:45
2   2026-05-13 12:19:17
dtype: datetime64[us]

After normalization:
0    2026-05-13
1    2026-05-13
2    2026-05-13
dtype: str

Date range: 2026-02-03 to 2026-05-13


,review_id,review,rating,date,bank,source
0,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13,CBE Bank,Google Play
1,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13,CBE Bank,Google Play
2,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13,CBE Bank,Google Play
3,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13,CBE Bank,Google Play
4,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",5,2026-05-13,CBE Bank,Google Play
...,...,...,...,...,...,...
695,fb955d85-3a17-4c39-80c8-a9d967ecab8f,"this is the worst update, what have you done w...",1,2026-02-03,CBE Bank,Google Play
696,947f41d3-8d10-4a9b-b499-785400609a39,how to open,5,2026-02-03,CBE Bank,Google Play
697,9e80832f-cddb-446d-891a-0cc61044cf83,I very much thanks again and again.,5,2026-02-03,CBE Bank,Google Play
698,918fc1bc-8054-4699-8f31-971e5a19930e,እየሰራ አይደለም,1,2026-02-03,CBE Bank,Google Play


#### Handle incorrect ratings

In [10]:
rating_distribution(df)

Rating distribution:
  5 stars:  471  ██████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:   59  ███████████
  3 stars:   45  █████████
  2 stars:   21  ████
  1 stars:  104  ████████████████████


In [11]:
validate_rating(df)

All ratings are valid (1-5).
Remaining: 700 reviews


,review_id,review,rating,date,bank,source
0,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13,CBE Bank,Google Play
1,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13,CBE Bank,Google Play
2,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13,CBE Bank,Google Play
3,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13,CBE Bank,Google Play
4,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",5,2026-05-13,CBE Bank,Google Play
...,...,...,...,...,...,...
695,fb955d85-3a17-4c39-80c8-a9d967ecab8f,"this is the worst update, what have you done w...",1,2026-02-03,CBE Bank,Google Play
696,947f41d3-8d10-4a9b-b499-785400609a39,how to open,5,2026-02-03,CBE Bank,Google Play
697,9e80832f-cddb-446d-891a-0cc61044cf83,I very much thanks again and again.,5,2026-02-03,CBE Bank,Google Play
698,918fc1bc-8054-4699-8f31-971e5a19930e,እየሰራ አይደለም,1,2026-02-03,CBE Bank,Google Play


#### Clean review text

In [12]:
df['review'] = df['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df['review'].head(10).to_string())

Sample cleaned reviews:
0                                        thank you cbe
1                                              is good
2                                                  wow
3                                     good application
4    nice, but i can't get some recently transactio...
5    very secure but very poor interface and limite...
6                                       very nice 100%
7                                                 good
8                                          good to use
9                                                  cbe


#### Save the cleaned dataset

In [13]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (700, 5)


,review,rating,date,bank,source
0,thank you cbe,5,2026-05-13,CBE Bank,Google Play
1,"nice, but i can't get some recently transactio...",5,2026-05-13,CBE Bank,Google Play
2,very nice 100%,5,2026-05-13,CBE Bank,Google Play
3,very secure but very poor interface and limite...,1,2026-05-13,CBE Bank,Google Play
4,is good,5,2026-05-13,CBE Bank,Google Play
5,good application,2,2026-05-13,CBE Bank,Google Play
6,wow,5,2026-05-13,CBE Bank,Google Play
7,good,5,2026-05-12,CBE Bank,Google Play
8,good to use,5,2026-05-12,CBE Bank,Google Play
9,cbe,1,2026-05-12,CBE Bank,Google Play


In [15]:
save_cleaned_data(df_clean, output_path="../data/processed/cbe_reviews_cleaned.csv")

Cleaned data saved to ../data/processed/cbe_reviews_cleaned.csv
Saved to: ../data/processed/cbe_reviews_cleaned.csv


### Report

In [17]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    700
  Reviews after cleaning :    700
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2026-02-03  to  2026-05-13

  Rating distribution:
    5 stars :  471 (67.3%)  ██████████████████████████████████████████████████████████████████████████████████████████████
    4 stars :   59 ( 8.4%)  ███████████
    3 stars :   45 ( 6.4%)  █████████
    2 stars :   21 ( 3.0%)  ████
    1 stars :  104 (14.9%)  ████████████████████

  Text length stats:
    Min    : 0 characters
    Median : 13 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

